# Study 889 — Broad Dollar-Hedge Overlay 🔁

**Is the currency-hedged-minus-unhedged gap for *broad* developed international the
US-vs-foreign rate differential — mechanically — and can you time a hedge on it?**

[Study 613](../../613-currency-hedged-etf-carry/) showed that for **one** market (Japan) the
return gap between a currency-*hedged* equity ETF and its *unhedged* twin is the covered-interest-
parity short-rate differential — "free carry hidden in a share class". Here we **generalise to
broad EAFE** (developed ex-US), where the differential is now *positive* (the Fed out-yields the
ECB/BoJ/BoE/SNB): does `hedged − unhedged` still equal the differential, and does a systematic
"hedge when the US out-yields" overlay add Sharpe?

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `7afa89fb9f2f`, as-of
2026-06-30); the live cells run the fast synthetic control. Young-ETF caveat: HEFA's clean tape
starts 2014-03 and the whole sample is one US-out-yields / rising-dollar era.*


## 1. What a hedged EAFE fund actually does

An *unhedged* EAFE fund (EFA) earns the local stocks **plus** whatever the euro, yen, pound and franc do against the dollar. A *hedged* fund (HEFA = EFA + one-month FX forwards) sells that currency basket forward and keeps only the local stocks — **plus** the forward's carry. Covered interest parity prices the forward at the rate gap, so the hedged fund quietly pockets `(r_US − r_foreign)`. Since 2022 the Fed sits *above* the EAFE central banks, so for a dollar holder the hedge now **pays**.

In [1]:
R = dict(h_carry=1.68, h_t=4.74, h_obs=1.35, h_beta=0.93, h_r2=0.91)
print('HEFA minus EFA, currency stripped out:')
print('  carry_hat = %+.2f%%/yr  (HAC t = %+.2f)' % (R['h_carry'], R['h_t']))
print('  observable US-EAFE policy differential: %+.2f%%/yr' % R['h_obs'])
print('  the hedge is a %.2f short of the currency basket (R2 = %.2f)'
        % (R['h_beta'], R['h_r2']))

HEFA minus EFA, currency stripped out:
  carry_hat = +1.68%/yr  (HAC t = +4.74)
  observable US-EAFE policy differential: +1.35%/yr
  the hedge is a 0.93 short of the currency basket (R2 = 0.91)


The carry the hedge pockets (**+1.68 %/yr**) sits right on the *observable* policy differential (**+1.35 %/yr**), and the hedge is a near-full short of the foreign currency basket (β = 0.93). Exactly 613's Japan mechanics — now broad, and dollar-favourable.

## 2. Is it just luck? A live synthetic control

We plant a known carry in a seeded toy world and check the estimator recovers it — and that it stays silent when the planted carry is zero. No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from dollar_hedge import data, strategy as st
null = st.synthetic_detect(data.synthetic_world(n_months=180, carry_annual=0.0, seed=889))
planted = st.synthetic_detect(data.synthetic_world(n_months=180, carry_annual=0.03, seed=889))
print('null world    : carry %+.2f%%/yr  HAC t = %+.2f  (should be ~0)'
      % (null['carry_ann_pct'], null['t_carry']))
print('planted (+3%%) : carry %+.2f%%/yr  HAC t = %+.2f  beta %.2f  (recovers it)'
      % (planted['carry_ann_pct'], planted['t_carry'], planted['beta']))

null world    : carry -0.44%/yr  HAC t = -1.22  (should be ~0)
planted (+3%) : carry +2.56%/yr  HAC t = +7.08  beta 1.00  (recovers it)


## 3. So can you *time* it? The honest answer: no — you just hold it

The obvious overlay is *hedge when the US out-yields, unhedge when it doesn't*. But the US has out-yielded EAFE for **93% of the post-2014 sample**, so the switch fires **1 time in twelve years** — it is effectively 'always hedge', and its brief unhedged spell only *lowers* the Sharpe below always-hedging (0.67 vs 0.75).

In [3]:
R = dict(ov_share=0.93, ov_switches=1, ov_sh=0.67, ov_hedged=0.75, ov_unhedged=0.4)
print('overlay Sharpe   %.2f  (switches: %d, share hedged %.0f%%)'
      % (R['ov_sh'], R['ov_switches'], R['ov_share']*100))
print('always-hedged    %.2f   <- just holding the hedged wrapper wins'  % R['ov_hedged'])
print('always-unhedged  %.2f' % R['ov_unhedged'])

overlay Sharpe   0.67  (switches: 1, share hedged 93%)
always-hedged    0.75   <- just holding the hedged wrapper wins
always-unhedged  0.40


## 4. The honest verdict

- **Signal — Real.** The 613 carry identity **generalises to broad EAFE**: HEFA minus EFA (currency stripped) is **+1.68 %/yr at HAC *t* = +4.74**, on the observable +1.35 %/yr differential, β = 0.93 short of the basket, holding in both eras. A genuine, dollar-favourable mechanical premium.
- **Tradability — Fragile.** It is real but you *hold* it, you don't *time* it: the US out-yields EAFE ~93% of the time so the switch adds nothing (0.67 vs 0.75 Sharpe), the raw hedged-sleeve win rests on one dollar regime, and the pure carry can't be cleanly isolated. Thin & un-timeable.